In [ ]:
# Install EasyDiffraction and prepare the example when running in Colab.
import importlib
import importlib.metadata
import importlib.util
import io
import os
import subprocess
import sys
import tempfile
import urllib.request
import zipfile
from pathlib import Path

installed = importlib.util.find_spec("edi") is not None
from_pip = bool(importlib.metadata.packages_distributions().get("edi"))
if "edi" not in sys.modules and (from_pip or not installed):
    pip = [sys.executable, "-m", "pip", "install", "--only-binary=:all:"]
    subprocess.check_call([
        *pip, "--isolated", "--no-deps", "--force-reinstall",
        "--index-url", "https://enhantica.github.io/edi/pypi/simple/", "easydiffraction",
    ])
    version = subprocess.check_output([
        sys.executable, "-c",
        "import importlib.metadata as m; print(m.version('easydiffraction'))",
    ], text=True).strip()
    subprocess.check_call([*pip, f"easydiffraction=={version}"])
    importlib.invalidate_caches()

if "google.colab" in sys.modules:
    workspace = Path(tempfile.mkdtemp(prefix="edi-tutorial-"))
    (workspace / "notebooks").mkdir()
    os.chdir(workspace / "notebooks")
    target = Path('../cli/sc-neut-cwl_tb2ti2o7-heidi/project').parent.resolve()
    target.mkdir(parents=True, exist_ok=True)
    url = "https://enhantica.github.io/edi/assets/tutorials/sc-neut-cwl_tb2ti2o7-heidi.zip"
    with urllib.request.urlopen(url, timeout=60) as response:
        data = response.read()
    with zipfile.ZipFile(io.BytesIO(data)) as bundle:
        for member in bundle.infolist():
            if not (target / member.filename).resolve().is_relative_to(target):
                raise ValueError("Unexpected path in tutorial bundle")
        bundle.extractall(target)


In [ ]:
"""Single-crystal fit: Tb₂Ti₂O₇, HEiDi."""

# Single-crystal fit: Tb₂Ti₂O₇, HEiDi

Fit the pyrochlore Tb₂Ti₂O₇ to constant-wavelength neutron single-crystal data from HEiDi at MLZ,
with anisotropic displacements and Becker-Coppens extinction.

The project is the catalogue case `sc-neut-cwl_tb2ti2o7-heidi`: the files the command line fits
with `python -m edi fit docs/user/cli/sc-neut-cwl_tb2ti2o7-heidi/project` and the app opens from
its Examples.

> This tutorial is not run when the documentation is built yet. It loads its project from the edi
> repository, so it runs once a tutorial runner can provide the project elsewhere
> (`execution-exclusions.yml`).

In [ ]:
import edi

## Load the project

In [ ]:
project = edi.Project.load('../cli/sc-neut-cwl_tb2ti2o7-heidi/project')
print(project.metadata.title)
print('Structures:', ', '.join(structure.name for structure in project.structures))
print('Experiments:', ', '.join(experiment.name for experiment in project.experiments))
print('Fitting mode:', project.fitting_mode)

## The start

The project starts from rough values: anisotropic displacements at a small isotropic-equivalent
value, the free O1 x moved off its refined value, mixed Becker-Coppens extinction with a crystal
radius of 3 mm and the domain radius (from 25 μm) and mosaic spread (from 0) free, and the scale
to one significant figure.

The fit is free to change these parameters:

In [ ]:
for parameter in project.free_parameters:
    print(f'{parameter.name:28} {parameter.value:12.6g} {parameter.units}')

## Fit

`analysis.fit()` runs the fit the project declares, here one experiment.

In [ ]:
result = project.analysis.fit()
print(f'{result.status.name.lower()} after {result.iterations} iterations')
print(f'reduced chi-square {result.reduced_chi_square:.3f}')

## Fitted values

In [ ]:
for label, value in result.values.items():
    print(f'{label:52} {value:12.6g} ± {result.uncertainty.get(label, 0):.2g}')

## Report

The HTML report lists the structure, the measured and calculated intensities and the fit. Open it
in a browser.

In [ ]:
project.report.save_html('sc-neut-cwl_tb2ti2o7-heidi.html')

## Save

`save_as` writes the project with its new values to a folder of its own; the catalogue's copy
stays as it was.

In [ ]:
project.save_as('sc-neut-cwl_tb2ti2o7-heidi')

## Explore the interactive report

Run the cell and trust the notebook to interact with the report. It includes Plotly plots and three.js structures where available.

In [ ]:
project.report.html_offline = True
project.report